# Level 1 TCN — Masked Multi-Scale Temporal Convolutional Network

This notebook implements and tests the **Level 1 Masked Multi-Scale TCN** architecture designed to overcome the padding, scaling, and temporal collapse issues that caused earlier deep learning models to underperform the Random Forest baseline.

**Key Features of this Architecture:**
- **Masked Temporal Convolutions:** Ignores padded timesteps properly.
- **Per-channel Normalization:** Handles the vastly different scales of IMU, ToF, and Thermopile sensors.
- **Residual Dilated Blocks:** Captures multi-scale temporal dynamics without losing gradient flow.
- **Masked Mean/Max Pooling:** Aggregates sequence features without diluting short sequences with padding zeros.

Local quick runs use `data/sample.csv`. Set `use_sample_data = False` for full `train.csv`.
Switch `search_mode` between `'grid'` and `'bayesian'` in the config cell.

In [1]:
import os
import sys
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

# Suppress TF and general warnings for cleaner output
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

# Local path routing (works from notebooks/ or project root)
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Kaggle path routing
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/src')
except Exception:
    pass

print('Paths configured. Workspace root:', workspace_root)

Paths configured. Workspace root: /kaggle/working


In [2]:
import tensorflow as tf
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score, make_scorer

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

try:
    from src import data_utils
    from src.base_utils_qwen import (
        SequenceExtractor,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
    )
    from src.tcn_level_one import MaskedMultiScaleTCNClassifier
    print('Imports loaded from src/')
except ImportError:
    import data_utils
    from base_utils_qwen import (
        SequenceExtractor,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
    )
    from tcn_level_one import MaskedMultiScaleTCNClassifier
    print('Imports loaded from flat src path')

# Install Bayesian search dependency if missing (safe to re-run)
if not SKOPT_AVAILABLE:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-optimize'])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
    print('Installed scikit-optimize')
else:
    import skopt
    print(f'scikit-optimize {skopt.__version__} ready')

Imports loaded from flat src path
scikit-optimize 0.10.2 ready


## 1. Configuration

Set your target, data source, and search parameters here.

In [3]:
TARGET_COL = 'bfrb'

# Use data/sample.csv for fast local runs; set False for full train.csv
use_sample_data = False
sample_file = 'sample.csv'

search_mode = 'bayesian'  # 'grid' or 'bayesian'
random_state = 42
n_splits = 1          # 1 -> GroupShuffleSplit; >=2 -> GroupKFold
cv_test_size = 0.5
train_size = 0.2
n_iter = 10 # Bayesian iterations (raise for full runs)
verbose = 4
error_const = 'raise'
patience = 10

results_dir = Path('results_tcn_level1')
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

if TARGET_COL == 'bfrb':
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average='macro', zero_division=0)

search_mode = str(search_mode).lower()
if search_mode in ('bayes', 'bayesian'):
    search_mode = 'bayesian'
elif search_mode != 'grid':
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=cv_test_size,
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)

print(f'Configuration set. Target: {TARGET_COL}, Search Mode: {search_mode}')

Configuration set. Target: bfrb, Search Mode: bayesian


## 2. Data Loading & Preprocessing

Load the raw row-level sensor data and engineer the target variables.

In [4]:
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file

if use_sample_data and sample_path.exists():
    raw_train_df = pd.read_csv(sample_path)
    print(f'Using {sample_file}: {raw_train_df["sequence_id"].nunique()} sequences')
else:
    raw_train_df = pd.read_csv(data_root / 'train.csv')
    print(f'Using train.csv: {raw_train_df["sequence_id"].nunique()} sequences')

# Handle demographics if available
demo_path = data_root / 'train_demographics.csv'
if demo_path.exists():
    train_demo_df = pd.read_csv(demo_path)

train_df = raw_train_df.set_index('row_id').copy(deep=True)

# Feature engineering for targets
train_df['gesture'] = train_df['gesture'].fillna('non_bfrb').astype(str)
train_df['orientation'] = train_df['orientation'].fillna('Unknown').astype(str)
train_df['is_target'] = train_df['sequence_type'].eq('Target').astype(int)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'].astype(bool), 'non_bfrb')

print(f'Data loaded. Shape: {train_df.shape}')
print(f'Target distribution:\n{train_df[TARGET_COL].value_counts()}')

Using Kaggle data folder: /kaggle/input/competitions/cmi-detect-behavior-with-sensor-data
Using train.csv: 8151 sequences
Data loaded. Shape: (574945, 342)
Target distribution:
bfrb
non_bfrb                    230887
Neck - scratch               56619
Eyebrow - pull hair          44305
Forehead - scratch           40923
Forehead - pull hairline     40802
Above ear - pull hair        40560
Neck - pinch skin            40507
Eyelash - pull hair          40218
Cheek - pinch skin           40124
Name: count, dtype: int64


## 3. Train / Holdout Split

We split by `sequence_id` to prevent temporal/sequence leakage.

In [5]:
try:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df,
        train_pct=train_size,
        test_pct=min(0.2, 1 - train_size),
        random_state=random_state,
    )
except Exception as e:
    print(f'Balanced split failed ({e}), falling back to GroupShuffleSplit.')
    seq_df = train_df[['sequence_id', 'is_target', TARGET_COL]].drop_duplicates('sequence_id').sort_values('sequence_id')
    gss = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = list(gss.split(seq_df, groups=seq_df['sequence_id']))[0]
    
    train_seqs = seq_df.iloc[train_idx]['sequence_id']
    test_seqs = seq_df.iloc[test_idx]['sequence_id']
    
    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)]
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)]

# For TCN, X is the raw row-level DataFrame, y contains the sequence-level metadata
X_train = train_sample_df.copy()
X_test = hold_out_df.copy()

y_train = X_train[['sequence_id', 'is_target', TARGET_COL]].copy()
y_test = X_test[['sequence_id', 'is_target', TARGET_COL]].copy()

groups = X_train['sequence_id'].astype(str)

print('Train sequences:', X_train['sequence_id'].nunique())
print('Test sequences:', X_test['sequence_id'].nunique())

Train: 1458 seqs | 17.9%
Test:  1452 seqs  | 17.8%
Train sequences: 1458
Test sequences: 1452


## 4. Model Definition

Initialize the Level 1 Masked Multi-Scale TCN. 

*Note: For smoke testing with `sample.csv`, we keep `epochs` low. Increase `epochs` to 50-100 for full training.*

In [6]:
extractor = SequenceExtractor(
    acc_modes='raw|velocity|jerk',
    rotation_modes='quaternion|angular_velocity',
    tof_modes='pooled_stats|sensor_stats',
    thm_modes='centered_diff',
    motion_filter_mode=None,
    use_dead_reckoning=False,
    compute_dt=True,
    interp_mode='linear',
    padding_value=0.0,
    chunk_window_size=None,
    chunk_stride=None,
    output_format='chunks',
    add_global_context=False,
    resample_modalities=False,
    stft_nperseg=32,
    stft_noverlap=None,
    stft_window_type='hann',
    stft_use_log_scale=True,
    cwt_wavelet='morl',
    cwt_max_scale=128,
    cwt_n_scales=32,
    cwt_use_log_scale=True,
    maxlen=200,
)

model = MaskedMultiScaleTCNClassifier(
    primary_target=TARGET_COL,
    sequence_col='sequence_id',
    extractor=extractor,
    filters=64,
    num_blocks=3,
    kernel_size=3,
    dilations=(1, 2, 4),
    dropout=0.2,
    learning_rate=1e-3,
    batch_size=32,
    epochs=50,                    # Smoke test epochs. Change to 50+ for full runs.
    patience=patience,
    validation_split=0.15,
    random_state=random_state,
    verbose=verbose,
    class_weight='balanced',
)

print('Level 1 TCN Model initialized.')

Level 1 TCN Model initialized.


## 5. Parameter Spaces

Define the search spaces for Grid and Bayesian optimization.

In [7]:
# ============================================================
# PARAMETER SPACE — GRID vs BAYESIAN
# ============================================================

_SKOPT_AVAILABLE = globals().get('SKOPT_AVAILABLE', False)
_ACTIVE_SEARCH_MODE = str(
    globals().get('SEARCH_MODE', globals().get('search_mode', 'grid'))
).lower()
if _ACTIVE_SEARCH_MODE in ('bayes', 'bayesian'):
    _ACTIVE_SEARCH_MODE = 'bayesian'

# Fast grid for sample.csv smoke tests
GRID_PARAM_SPACE_QUICK = {
    'extractor__acc_modes': ['raw|velocity|jerk'],
    'extractor__rotation_modes': ['quaternion|angular_velocity'],
    'extractor__tof_modes': ['pooled_stats|sensor_stats'],
    'extractor__thm_modes': ['centered_diff'],
    'extractor__motion_filter_mode': [None],
    'extractor__use_dead_reckoning': [False],
    'extractor__dead_reckoning_detrend': [False],
    'extractor__kalman_process_noise': [1e-3],
    'extractor__kalman_measurement_noise': [1e-1],
    'extractor__window_size': [7],
    'extractor__smooth_alpha': [None],
    'extractor__clip_value': [None],
    'extractor__interp_mode': ['linear'],
    'extractor__output_format': ['chunks'],
    'extractor__padding_value': [0.0],
    'extractor__maxlen': [160, 200],
    'extractor__chunk_window_size': [None],
    'extractor__chunk_stride': [None],
    'extractor__add_global_context': [False],
    'extractor__resample_modalities': [False],
    'extractor__compute_dt': [True],
    'extractor__imu_native_sampling_rate': [20],
    'extractor__rot_native_sampling_rate': [20],
    'extractor__tof_native_sampling_rate': [5],
    'extractor__thm_native_sampling_rate': [5],
    'extractor__stft_nperseg': [32],
    'extractor__stft_noverlap': [None],
    'extractor__stft_window_type': ['hann'],
    'extractor__stft_use_log_scale': [True],
    'extractor__cwt_wavelet': ['morl'],
    'extractor__cwt_max_scale': [64, 128],
    'extractor__cwt_n_scales': [16, 32],
    'extractor__cwt_use_log_scale': [True],
    'filters': [32, 64],
    'num_blocks': [2, 3],
    'kernel_size': [3],
    'dilations': [(1, 2, 4), (1, 2, 4, 8)],
    'dropout': [0.2, 0.3],
    'learning_rate': [1e-3, 1e-4],
    'batch_size': [32],
}

# Practical compact grid for full train.csv
GRID_PARAM_SPACE = {
    'extractor__acc_modes': [
        'raw|velocity|jerk',
        'smoothed|velocity|displacement|jerk',
    ],
    'extractor__rotation_modes': [
        'quaternion|angular_velocity',
        'quaternion|euler|angular_velocity',
    ],
    'extractor__tof_modes': [
        'pooled_stats|sensor_stats',
        'pooled_stats',
    ],
    'extractor__thm_modes': [
        'centered_diff',
        'centered',
    ],
    'extractor__motion_filter_mode': [None],
    'extractor__use_dead_reckoning': [False],
    'extractor__dead_reckoning_detrend': [False],
    'extractor__kalman_process_noise': [1e-3],
    'extractor__kalman_measurement_noise': [1e-1],
    'extractor__window_size': [7],
    'extractor__smooth_alpha': [None],
    'extractor__clip_value': [None],
    'extractor__interp_mode': ['linear'],
    'extractor__output_format': ['chunks'],
    'extractor__padding_value': [0.0],
    'extractor__maxlen': [160, 200],
    'extractor__chunk_window_size': [None, 80],
    'extractor__chunk_stride': [None, 20],
    'extractor__add_global_context': [False],
    'extractor__resample_modalities': [False],
    'extractor__compute_dt': [True],
    'extractor__imu_native_sampling_rate': [20],
    'extractor__rot_native_sampling_rate': [20],
    'extractor__tof_native_sampling_rate': [5],
    'extractor__thm_native_sampling_rate': [5],
    'extractor__stft_nperseg': [32, 64],
    'extractor__stft_noverlap': [None, 16],
    'extractor__stft_window_type': ['hann', 'hamming'],
    'extractor__stft_use_log_scale': [True, False],
    'extractor__cwt_wavelet': ['morl', 'mexh'],
    'extractor__cwt_max_scale': [64, 128],
    'extractor__cwt_n_scales': [16, 32],
    'extractor__cwt_use_log_scale': [True, False],
    'filters': [64, 128],
    'num_blocks': [3, 4],
    'kernel_size': [3, 5],
    'dilations': [(1, 2, 4), (1, 2, 4, 8), (1, 4, 16)],
    'dropout': [0.1, 0.2, 0.3],
    'learning_rate': [1e-3, 5e-4, 1e-4],
    'batch_size': [32, 64],
}

if _SKOPT_AVAILABLE:
    try:
        BAYESIAN_PARAM_SPACE = {
            'extractor__acc_modes': Categorical([
                'raw|velocity',
                'raw|velocity|jerk',
                'smoothed|velocity|displacement|jerk',
            ]),
            'extractor__rotation_modes': Categorical([
                'quaternion|euler|angular_velocity',
                'quaternion|delta_euler|angular_velocity',
                'quaternion|angular_velocity|delta_euler|rot6d',
            ]),
            'extractor__tof_modes': Categorical([
                'pooled_stats|sensor_stats',
            ]),
            'extractor__thm_modes': Categorical([
                'centered_diff',
            ]),
            'extractor__motion_filter_mode': Categorical([
                'extended_kalman',
            ]),
            'extractor__use_dead_reckoning': Categorical([ True]),
            'extractor__dead_reckoning_detrend': Categorical([True]),
            'extractor__kalman_process_noise': Real(1e-5, 1e-1, prior='log-uniform'),
            'extractor__kalman_measurement_noise': Real(1e-3, 1e1, prior='log-uniform'),
            'extractor__window_size': Integer(10, 20),
            'extractor__smooth_alpha': Categorical([None]),
            'extractor__clip_value': Categorical([None]),
            'extractor__interp_mode': Categorical(['linear']),
            'extractor__output_format': Categorical(['chunks']),
            'extractor__padding_value': Categorical([0.0]),
            'extractor__maxlen': Categorical([150]),
            'extractor__chunk_window_size': Categorical([100]),
            'extractor__chunk_stride': Categorical([50]),
            'extractor__add_global_context': Categorical([True]),
            'extractor__compute_dt': Categorical([True]),
            'extractor__imu_native_sampling_rate': Categorical([100]),
            'extractor__rot_native_sampling_rate': Categorical([100]),
            'extractor__tof_native_sampling_rate': Categorical([20]),
            'extractor__thm_native_sampling_rate': Categorical([20]),
            'extractor__imu_target_sampling_rate': Categorical([100]),
            'extractor__rot_target_sampling_rate': Categorical([100]),
            'extractor__tof_target_sampling_rate': Categorical([20]),
            'extractor__thm_target_sampling_rate': Categorical([20]),
            'extractor__resample_modalities': Categorical([True]),
            'extractor__stft_nperseg': Integer(16, 64),
            'extractor__stft_noverlap': Categorical([None, 4, 8, 12]),
            'extractor__stft_window_type': Categorical(['hann', 'hamming']),
            'extractor__stft_use_log_scale': Categorical([True, False]),
            'extractor__cwt_wavelet': Categorical(['morl', 'mexh']),
            'extractor__cwt_max_scale': Integer(32, 128),
            'extractor__cwt_n_scales': Integer(8, 32),
            'extractor__cwt_use_log_scale': Categorical([True, False]),
            'filters': Integer(64, 512),
            'num_blocks': Integer(2, 6),
            'kernel_size': Categorical([3, 5]),
            'dilations': Categorical([(1, 2, 4)]),
            'dropout': Real(0.1, 0.15),
            'learning_rate': Real(1e-3, 5e-3, prior='log-uniform'),
            'batch_size': Categorical([64, 128]),
        }
    except Exception:
        BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE
else:
    BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE

if _ACTIVE_SEARCH_MODE == 'bayesian' and _SKOPT_AVAILABLE:
    param_space = BAYESIAN_PARAM_SPACE
    try:
        param_space = prepare_bayesian_space(param_space)
    except Exception:
        pass
elif use_sample_data:
    param_space = GRID_PARAM_SPACE_QUICK
else:
    param_space = GRID_PARAM_SPACE

if _ACTIVE_SEARCH_MODE == 'grid':
    grid_size = 1
    for values in param_space.values():
        grid_size *= len(values)
    print('Grid combinations:', grid_size)
else:
    print('Bayesian iterations:', n_iter)

print('Active search mode:', _ACTIVE_SEARCH_MODE)
print('Parameter keys:', len(param_space))

Bayesian iterations: 10
Active search mode: bayesian
Parameter keys: 44


## 6. Search & Training Execution

Run the hyperparameter search. This will train multiple TCN configurations.

In [8]:
if search_mode == 'bayesian':
    if not SKOPT_AVAILABLE:
        raise ImportError(
            "Bayesian search requires scikit-optimize. "
            "Install with: pip install scikit-optimize"
        )

    search = BayesSearchCV(
        estimator=model,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        error_score=error_const,
    )
else:
    search = GridSearchCV(
        estimator=model,
        param_grid=param_space,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        verbose=verbose,
        return_train_score=True,
        error_score=error_const,
    )

print(f'Starting {search_mode.upper()} search...')
search.fit(X_train, y_train, groups=groups)

print('\nBest CV score:', search.best_score_)
print('Best params:', search.best_params_)

Starting BAYESIAN search...
Fitting 1 folds for each of 1 candidates, totalling 1 fits


I0000 00:00:1789758309.065241      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1789758309.068123      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/50


I0000 00:00:1789758345.621049      68 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Epoch 2/50
Epoch 3/50
Epoch 4/50
Epoch 5/50
Epoch 6/50
Epoch 7/50
Epoch 8/50
Epoch 9/50
Epoch 10/50
Epoch 11/50
Epoch 12/50
Epoch 13/50
Epoch 14/50
Epoch 15/50
Epoch 16/50
Epoch 17/50
Epoch 18/50
Epoch 19/50
Epoch 20/50
Epoch 21/50
Epoch 22/50
Epoch 22: early stopping
Restoring model weights from the end of the best epoch: 12.
[CV 1/1] END batch_size=64, dilations=[1, 2, 4], dropout=0.1466433999423917, extractor__acc_modes=raw|velocity, extractor__add_global_context=True, extractor__chunk_stride=50, extractor__chunk_window_size=100, extractor__clip_value=None, extractor__compute_dt=True, extractor__cwt_max_scale=94, extractor__cwt_n_scales=21, extractor__cwt_use_log_scale=True, extractor__cwt_wavelet=morl, extractor__dead_reckoning_detrend=True, extractor__imu_native_sampling_rate=100, extractor__imu_target_sampling_rate=100, extractor__interp_mode=linear, extractor__kalman_measurement_noise=1.2244720462710987, extractor__kalman_process_noise=1.4587470230927182e-05, extractor__maxlen=1

## 7. Holdout Evaluation & Saving Results

In [9]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

eval_results = evaluate_holdout(
    y_test,
    y_pred,
    target_col=TARGET_COL,
    verbose=True,
)

print('Holdout competition score:', eval_results['competition_score'])

cv_df = pd.DataFrame(search.cv_results_)
cv_df.to_csv(results_dir / f'tcn_level1_cv_{timestamp}.csv', index=False)

eval_results['results_df'].to_csv(
    results_dir / f'tcn_level1_holdout_{timestamp}.csv',
    index=False,
)

pd.DataFrame(
    [
        {
            'best_score': search.best_score_,
            'best_params': str(search.best_params_),
            'holdout_score': eval_results['competition_score'],
        }
    ]
).to_csv(
    results_dir / f'tcn_level1_best_{timestamp}.csv',
    index=False,
)

model_path = results_dir / f'tcn_level1_model_{timestamp}.keras'
try:
    if hasattr(best_model, 'model_') and best_model.model_ is not None:
        best_model.model_.save(model_path)
        print(f'TF model saved to {model_path}')
except Exception as e:
    print(f'Could not save TF model directly: {e}')


FINAL EVALUATION
Binary F1 (non_bfrb vs bfrb): 0.9048
BFRB Gesture Macro F1: 0.2005
COMPETITION SCORE: 0.5526

----------------------------------------
BFRB Gesture Classification Report
----------------------------------------
                          precision    recall  f1-score   support

   Above ear - pull hair       0.23      0.56      0.32        81
      Cheek - pinch skin       0.39      0.15      0.21        81
     Eyebrow - pull hair       0.19      0.17      0.18        81
     Eyelash - pull hair       0.23      0.14      0.17        81
Forehead - pull hairline       0.00      0.00      0.00        81
      Forehead - scratch       0.34      0.54      0.42        81
       Neck - pinch skin       0.24      0.32      0.28        81
          Neck - scratch       0.50      0.14      0.21        81
                non_bfrb       0.00      0.00      0.00         0

                accuracy                           0.25       648
               macro avg       0.24      0.

## 8. Action Items for Full Training

1. **Increase Epochs:** Change `epochs=5` to `epochs=50` (or higher) in the Model Definition cell.
2. **Full Data:** Set `use_sample_data = False` to use `train.csv`.
3. **Search Mode:** Switch `search_mode = 'bayesian'` and increase `n_iter = 30` (or use `search_mode = 'grid'` with `GRID_PARAM_SPACE`).
4. **Sequence Window:** Tune `extractor__maxlen` / `extractor__chunk_window_size` so they cover the 95th percentile of sequence lengths in `train.csv`.
5. **Feature Extraction:** Bayesian search explores the full extractor space; grid search uses the compact `GRID_PARAM_SPACE` subset.